# Chapter 6: Generative AI

**Welcome to Chapter 6**. This notebook contains the listings for Chapter 6, which explains the fundamentals of Deep Learning in PyTorch.

# Listing 6-1: Generating Crumpled Paper with a VAE


This listing implements a subset of the general skeleton for the end-to-end lifecycle of a PyTorch project, which includes loading and preparing data, defining or loading a model, specifying the loss function and optimizer, training with validation, evaluating performance, and saving the model for later use. This code demonstrates the essential stages of a working PyTorch program without including optional steps such as inference pipelines or model deployment.

In [ ]:
# --------------------------------------------------
# Step 1: Install dependencies
# --------------------------------------------------
!pip install -q trimesh matplotlib

# --------------------------------------------------
# Step 2: Imports and Device Setup
# --------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import trimesh
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# Step 3: Crumpled Paper Dataset (Perturbed Spheres)
# --------------------------------------------------
class SphereMeshDataset(Dataset):
    def __init__(self, num_samples=300, subdivisions=2):
        self.mesh_template = trimesh.creation.icosphere(subdivisions=subdivisions)
        self.num_samples = num_samples
        self.faces = self.mesh_template.faces

    def __len__(self):
        return self.num_samples

    def __getitem__(self, _):
        scale = np.random.uniform(0.04, 0.09)
        perturbation = np.random.normal(scale=scale, size=self.mesh_template.vertices.shape)
        verts = self.mesh_template.vertices + perturbation
        return torch.tensor(verts.flatten(), dtype=torch.float32)

# --------------------------------------------------
# Step 4: VAE Architecture and Reparameterization
# --------------------------------------------------
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU()
        )
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

# --------------------------------------------------
# Step 5: Loss Function
# --------------------------------------------------
def vae_loss(recon, x, mu, logvar):
    recon_loss = F.mse_loss(recon, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl

# --------------------------------------------------
# Step 6: Training the VAE
# --------------------------------------------------
dataset = SphereMeshDataset()
input_dim = dataset[0].numel()
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
vae = VAE(input_dim=input_dim, latent_dim=16).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(500):
    total = 0
    for batch in dataloader:
        batch = batch.to(device)
        recon, mu, logvar = vae(batch)
        loss = vae_loss(recon, batch, mu, logvar)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total:.2f}")

# --------------------------------------------------
# Step 7: Inference and Sampling
# --------------------------------------------------
vae.eval()
with torch.no_grad():
    faces = dataset.mesh_template.faces
    example = next(iter(dataloader))[0].unsqueeze(0).to(device)
    recon, mu, logvar = vae(example)

    var1 = vae.decoder(vae.reparameterize(mu, logvar))
    var2 = vae.decoder(vae.reparameterize(mu, logvar))
    var3 = vae.decoder(vae.reparameterize(mu, logvar))

    def unflatten(tensor): return tensor.cpu().numpy().reshape(-1, 3)

    original_verts = unflatten(example)
    recon_verts = unflatten(recon)
    var1_verts = unflatten(var1)
    var2_verts = unflatten(var2)
    var3_verts = unflatten(var3)

# --------------------------------------------------
# Step 8: Visualization
# --------------------------------------------------
def plot_meshes(verts_list, faces, titles):
    fig, axs = plt.subplots(1, 5, figsize=(20,5), subplot_kw={'projection': '3d'})
    paper_yellow = np.array([1.0, 1.0, 0.85])

    for ax, verts, title in zip(axs, verts_list, titles):
        mesh = Poly3DCollection(verts[faces], facecolors=paper_yellow, edgecolors='black', linewidths=0.2, alpha=1.0)
        ax.add_collection3d(mesh)
        ax.set_xlim(-1.2, 1.2)
        ax.set_ylim(-1.2, 1.2)
        ax.set_zlim(-1.2, 1.2)
        ax.axis('off')
        ax.view_init(elev=20, azim=30)
        ax.set_title(title, pad=5, fontsize=22)

    plt.show()

plot_meshes(
    [original_verts, recon_verts, var1_verts, var2_verts, var3_verts],
    faces,
    ["Original", "Reconstruction", "Variation 1", "Variation 2", "Variation 3"]
)

# Listing 6-2: Generating Realistic Faces with StyleGAN2-ADA
This example shows how to use StyleGAN2-ADA to generate realistic synthetic faces.

In [ ]:
# --------------------------------------------------
# Step 1. Clone StyleGAN2-ADA and Install Dependencies
# --------------------------------------------------
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch

# --------------------------------------------------
# Step 2. Install dependencies
# --------------------------------------------------
import random
!pip install ninja pyspng imageio-ffmpeg==0.4.3

# --------------------------------------------------
# Step 3. Download Pretrained FFHQ Model (Faces)
# --------------------------------------------------
!mkdir pretrained
!wget -O pretrained/ffhq.pkl https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl

# --------------------------------------------------
# Step 4. Let Python choose random seeds for you
# --------------------------------------------------
a = random.randint(0, 7000)
b = a + 11 # Generate 1 + 11 = 12 faces

# --------------------------------------------------
# Step 5. Generate Faces from Pretrained Model
# --------------------------------------------------
!python generate.py \
  --outdir=generated_faces \
  --trunc=0.7 \
  --seeds={a}-{b} \
  --network=pretrained/ffhq.pkl

# --------------------------------------------------
# Step 6. View the Generated Faces
# --------------------------------------------------
from IPython.display import Image, display
import os

for filename in sorted(os.listdir('generated_faces')):
    if filename.endswith('.png'):
       display(Image(filename=f'generated_faces/{filename}'))


# Listing 6-3: GAN Training Example
This code demonstrates how GANs implement the training process.

In [ ]:
# --------------------------------------------------
# Step 1: Install dependencies
# --------------------------------------------------
!pip install -q torch torchvision matplotlib

# --------------------------------------------------
# Step 2: Imports, device, and reproducibility
# --------------------------------------------------
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------------
# Step 3: Data (MNIST normalized to [-1, 1])
# --------------------------------------------------
batch_size = 128
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

# --------------------------------------------------
# Step 4: Model definitions
# --------------------------------------------------
z_dim = 100
img_dim = 28 * 28

class Generator(nn.Module):
    def __init__(self, z_dim, img_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, img_dim),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, img_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

G = Generator(z_dim, img_dim).to(device)
D = Discriminator(img_dim).to(device)

# --------------------------------------------------
# Step 5: Loss and optimizers
# --------------------------------------------------
criterion = nn.BCELoss()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

# --------------------------------------------------
# Step 6: Training loop
# --------------------------------------------------
epochs = 20
fixed_z = torch.randn(64, z_dim, device=device)

d_losses = []
g_losses = []

for epoch in range(1, epochs + 1):

    d_running = 0.0
    g_running = 0.0
    n_batches = 0

    for real, _ in train_loader:
        n_batches += 1
        real = real.to(device).view(-1, img_dim)
        bsz = real.size(0)

        real_labels = torch.ones(bsz, 1, device=device)
        fake_labels = torch.zeros(bsz, 1, device=device)

        # -----------------------------
        # Step 6a: Train Discriminator
        # -----------------------------

        opt_D.zero_grad(set_to_none=True)

        # 6a.i Real samples
        real_pred = D(real)
        d_loss_real = criterion(real_pred, real_labels)

        # 6a.ii Fake samples
        z = torch.randn(bsz, z_dim, device=device)
        fake = G(z)
        fake_pred = D(fake.detach())
        d_loss_fake = criterion(fake_pred, fake_labels)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        opt_D.step()

        # -----------------------------
        # Step 6b: Train Generator
        # -----------------------------

        opt_G.zero_grad(set_to_none=True)

        z = torch.randn(bsz, z_dim, device=device)
        fake = G(z)
        fake_pred = D(fake)

        # Non-saturating objective
        g_loss = criterion(fake_pred, real_labels)

        g_loss.backward()
        opt_G.step()

        d_running += d_loss.item()
        g_running += g_loss.item()

    d_epoch = d_running / n_batches
    g_epoch = g_running / n_batches

    d_losses.append(d_epoch)
    g_losses.append(g_epoch)

    print(f"Epoch {epoch}/{epochs} | D Loss: {d_epoch:.4f} | G Loss: {g_epoch:.4f}")

    # --------------------------------------------------
    # Step 7: Save sample grid
    # --------------------------------------------------
    with torch.no_grad():
        G.eval()
        samples = G(fixed_z).view(-1, 1, 28, 28)
        grid = utils.make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
        utils.save_image(grid, f"epoch_{epoch:02d}.png")

# --------------------------------------------------
# Step 8: Plot losses (Figure 6-4)
# --------------------------------------------------
epochs_arr = np.arange(1, epochs + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_arr, d_losses, marker="o", linestyle="None", label="Discriminator (D)")
plt.plot(epochs_arr, g_losses, marker="x", linestyle="None", label="Generator (G)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Listing 6-4 Training and Fine-Tuning a GAN (Circles and Slashes)

In [ ]:
# --------------------------------------------------
# Step 1: Imports and Device Setup
# --------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# --------------------------------------------------
# Step 2: Custom Shape Dataset
# --------------------------------------------------
class ShapeDataset(Dataset):
    def __init__(self, count=1000, shape="circle"):
        self.images = []

        for _ in range(count):
            img = Image.new("L", (28, 28), color=0)
            draw = ImageDraw.Draw(img)

            if shape == "circle":
                r = 8
                center = (14, 14)
                box = [center[0] - r, center[1] - r,
                       center[0] + r, center[1] + r]
                draw.ellipse(box, fill=255)

            elif shape == "slash":
                draw.line([5, 23, 23, 5], fill=255, width=3)

            # Normalize pixels to [-1, 1] to match Tanh output
            array = (np.array(img, dtype=np.float32) / 127.5) - 1.0
            self.images.append(torch.tensor(array).unsqueeze(0))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx]

# --------------------------------------------------
# Step 3: Generator and Discriminator
# --------------------------------------------------
Z_DIM = 100

class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# --------------------------------------------------
# Step 4: Sample Visualization Helper
# --------------------------------------------------
def show_samples(generator, title, n=16):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n, Z_DIM, device=device)
        samples = generator(z).cpu()

    fig, axs = plt.subplots(2, 8, figsize=(10, 3))
    for i, ax in enumerate(axs.flat):
        img = (samples[i, 0] + 1) / 2  # map [-1, 1] -> [0, 1]
        ax.imshow(img, cmap="gray")
        ax.axis("off")

    plt.suptitle(title)
    plt.show()
    generator.train()

# --------------------------------------------------
# Step 5: Training Function
# --------------------------------------------------
def train_gan(
    generator,
    discriminator,
    dataloader,
    epochs,
    title_prefix="",
    lr=2e-4,
    freeze_g_layers: int = 0,
    use_differential_g_lr: bool = False,
    g_lr_low: float = 1e-5,
    g_lr_high: float = 1e-4,
):
    """
    freeze_g_layers:
        Number of early modules in generator.net to freeze (0 = no freezing).
        This is a simple control knob for the 'freeze early layers' idea.

    use_differential_g_lr:
        If True, use different learning rates for earlier vs later parts of G.
        This matches the "Differential Learning Rates" tip in the text.
    """
    criterion = nn.BCELoss()

    # ------------------------------------------------
    # Alternative Step 5A: Selective Fine-Tuning (Freeze early G layers)
    # ------------------------------------------------
    # Freeze the first `freeze_g_layers` modules inside generator.net.
    # Example: freeze_g_layers=2 freezes the first Linear+ReLU pair.
    if freeze_g_layers > 0:
        modules = list(generator.net.children())
        for m in modules[:freeze_g_layers]:
            for p in m.parameters():
                p.requires_grad = False

    # ------------------------------------------------
    # Step 5B: Optimizers
    # ------------------------------------------------
    # Default: single learning rate for all generator parameters
    if not use_differential_g_lr:
        opt_G = optim.Adam(
            filter(lambda p: p.requires_grad, generator.parameters()),
            lr=lr,
            betas=(0.5, 0.999),
        )
    else:
        # ------------------------------------------------
        # Alternative Step 5B (Option): Differential Learning Rates for G
        # ------------------------------------------------
        # Split generator.net into "earlier" and "later" modules.
        # The last two modules are Linear(256->784) and Tanh in this example.
        g_children = list(generator.net.children())
        early_modules = g_children[:-2]
        late_modules = g_children[-2:]

        early_params = []
        for m in early_modules:
            early_params += [p for p in m.parameters() if p.requires_grad]

        late_params = []
        for m in late_modules:
            late_params += [p for p in m.parameters() if p.requires_grad]

        opt_G = optim.Adam(
            [
                {"params": early_params, "lr": g_lr_low},
                {"params": late_params, "lr": g_lr_high},
            ],
            betas=(0.5, 0.999),
        )

    opt_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

    # ------------------------------------------------
    # Step 5C: Training Loop
    # ------------------------------------------------
    for epoch in range(epochs):
        for real_imgs in dataloader:
            real_imgs = real_imgs.to(device)
            batch = real_imgs.size(0)

            real_labels = torch.ones(batch, 1, device=device)
            fake_labels = torch.zeros(batch, 1, device=device)

            # -------------------
            # Step 5C.i: Discriminator update
            # -------------------
            z = torch.randn(batch, Z_DIM, device=device)
            fake_imgs = generator(z)

            loss_D = (
                criterion(discriminator(real_imgs), real_labels) +
                criterion(discriminator(fake_imgs.detach()), fake_labels)
            )

            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

            # -------------------
            # Step 5C.ii: Generator update (non-saturating objective)
            # -------------------
            loss_G = criterion(discriminator(fake_imgs), real_labels)

            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

        print(
            f"{title_prefix} Epoch {epoch+1}: "
            f"Loss_D={loss_D.item():.4f}, "
            f"Loss_G={loss_G.item():.4f}"
        )

        show_samples(generator, f"{title_prefix} Epoch {epoch+1}")

# --------------------------------------------------
# Step 6: Train on Circles
# --------------------------------------------------
G = Generator().to(device)
D = Discriminator().to(device)

circle_loader = DataLoader(
    ShapeDataset(shape="circle"),
    batch_size=64,
    shuffle=True
)

train_gan(
    G,
    D,
    circle_loader,
    epochs=246,
    title_prefix="CIRCLE"
)

# --------------------------------------------------
# Step 7: Fine-Tune on Slashes
# --------------------------------------------------
slash_loader = DataLoader(
    ShapeDataset(shape="slash"),
    batch_size=64,
    shuffle=True
)

train_gan(
    G,
    D,
    slash_loader,
    epochs=100,
    title_prefix="FINE-TUNE (SLASHES)",
    lr=1e-4,                 # reduced LR for stability
    freeze_g_layers=2,        # Alternative Step: freeze early G layers (optional)
    use_differential_g_lr=False  # set True to enable differential G learning rates
)

# Listing 6-5: Diffusion Model Example

In [ ]:
# --------------------------------------------------
# Step 1. Install and Import Libraries
# --------------------------------------------------
!pip install -q torch matplotlib

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# Step 2. Create a Synthetic Dataset of Sine Waves
# --------------------------------------------------
class SineDataset(Dataset):
    def __init__(self, n=20000, length=128):
        self.n = n
        self.length = length
        self.x = torch.linspace(0, 1, steps=length)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        amp = torch.empty(1).uniform_(0.5, 1.5).item()
        freq = torch.empty(1).uniform_(1.0, 6.0).item()
        phase = torch.empty(1).uniform_(0, 2 * math.pi).item()
        y = amp * torch.sin(2 * math.pi * freq * self.x + phase)
        return y.unsqueeze(0)  # shape: [1, length]

# --------------------------------------------------
# Step 3. Define the Diffusion Noise Schedule
# --------------------------------------------------
T = 200  # diffusion steps

betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# --------------------------------------------------
# Step 4. Implement Forward Diffusion
# --------------------------------------------------
def q_sample(x0, t, noise=None):
    """
    x0: [B, 1, L]
    t:  [B] integer timesteps in [0, T-1]
    """
    if noise is None:
        noise = torch.randn_like(x0)
    a_bar = alpha_bars[t].view(-1, 1, 1)
    xt = torch.sqrt(a_bar) * x0 + torch.sqrt(1 - a_bar) * noise
    return xt, noise

# --------------------------------------------------
# Step 5. Define the Noise-Prediction Network
# --------------------------------------------------
class Tiny1DUNet(nn.Module):
    def __init__(self, length=128, time_dim=64):
        super().__init__()
        self.time_emb = nn.Sequential(
            nn.Embedding(T, time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.conv1 = nn.Conv1d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv1d(64, 32, 3, padding=1)
        self.conv4 = nn.Conv1d(32, 1, 3, padding=1)

        self.fc_t = nn.Linear(time_dim, 64)

    def forward(self, x, t):
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))

        te = self.time_emb(t)            # [B, time_dim]
        te = self.fc_t(te).unsqueeze(-1) # [B, 64, 1]
        h = h + te                       # timestep conditioning

        h = F.relu(self.conv3(h))
        return self.conv4(h)

# --------------------------------------------------
# Step 6. Train the Model
# --------------------------------------------------
print("Training 1D diffusion model on sine waves...")
print("=" * 60)

dataset = SineDataset(n=20000, length=128)
loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)

model = Tiny1DUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

model.train()
for epoch in range(100):
    total = 0.0

    for x0 in loader:
        x0 = x0.to(device)

        # sample a random timestep for each example
        t = torch.randint(0, T, (x0.size(0),), device=device)

        # corrupt the clean signal to timestep t
        xt, noise = q_sample(x0, t)

        # predict the added noise
        pred = model(xt, t)

        # compare predicted noise to true noise
        loss = F.mse_loss(pred, noise)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: loss={total / len(loader):.4f}")

print("Training complete!")

# --------------------------------------------------
# Step 7. Sample New Signals (Reverse Diffusion)
# --------------------------------------------------
@torch.no_grad()
def p_sample_loop(model, shape):
    """DDPM-style reverse diffusion sampling."""
    model.eval()
    x = torch.randn(shape, device=device)  # start from pure noise

    for t in reversed(range(T)):
        tt = torch.full((shape[0],), t, device=device, dtype=torch.long)

        a = alphas[t]
        a_bar = alpha_bars[t]
        b = betas[t]

        # predict the noise currently present
        eps = model(x, tt)

        # compute the reverse-process mean estimate
        coef1 = 1 / torch.sqrt(a)
        coef2 = b / torch.sqrt(1 - a_bar)
        mean = coef1 * (x - coef2 * eps)

        # add stochasticity except at the final step
        if t > 0:
            z = torch.randn_like(x)
            sigma = torch.sqrt(b)
            x = mean + sigma * z
        else:
            x = mean

    return x

samples = p_sample_loop(model, shape=(8, 1, 128)).cpu()

# --------------------------------------------------
# Step 8. Visualize Results
# --------------------------------------------------
print("\nCreating visualization...")

# one clean sine wave
original = dataset[0].to(device).unsqueeze(0)  # [1, 1, 128]

# forward-diffusion examples at several timesteps
timesteps = [0, 50, 100, 150, 199]
noisy_waves = []
for t_val in timesteps:
    t = torch.tensor([t_val], device=device)
    noisy, _ = q_sample(original, t)
    noisy_waves.append(noisy)

# generate several new samples
num_samples = 5
generated = p_sample_loop(model, shape=(num_samples, 1, 128))

# plot everything
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 6, hspace=0.4, wspace=0.3)

# original signal
ax_orig = fig.add_subplot(gs[0, :2])
ax_orig.plot(original[0, 0].cpu().numpy(), linewidth=2, color="blue")
ax_orig.set_title("Original Sine Wave", fontsize=14, fontweight="bold")
ax_orig.set_xlabel("Time")
ax_orig.set_ylabel("Amplitude")
ax_orig.grid(True, alpha=0.3)

# progressively noisier versions
for i, (t_val, noisy) in enumerate(zip(timesteps[1:], noisy_waves[1:])):
    ax = fig.add_subplot(gs[0, i + 2])
    ax.plot(noisy[0, 0].cpu().numpy(), linewidth=1.5, alpha=0.8)
    ax.set_title(f"Noise t={t_val}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.set_ylabel("Amplitude")

# final noisy example
ax_noise = fig.add_subplot(gs[1, 0])
ax_noise.plot(noisy_waves[-1][0, 0].cpu().numpy(), linewidth=1.5, alpha=0.8, color="gray")
ax_noise.set_title("Highly Noisy Input", fontsize=12, fontweight="bold")
ax_noise.set_xlabel("Time")
ax_noise.set_ylabel("Amplitude")
ax_noise.grid(True, alpha=0.3)

# arrow / transition
ax_arrow = fig.add_subplot(gs[1, 1])
ax_arrow.text(0.5, 0.5, "→\nDenoise\n→", ha="center", va="center",
              fontsize=16, fontweight="bold", transform=ax_arrow.transAxes)
ax_arrow.axis("off")

# generated samples
colors = ["red", "green", "purple", "orange", "brown"]
for i in range(num_samples):
    if i < 4:
        ax = fig.add_subplot(gs[1, i + 2])
    else:
        ax = fig.add_subplot(gs[2, 1])

    ax.plot(generated[i, 0].cpu().numpy(), linewidth=2, alpha=0.9, color=colors[i])
    ax.set_title(f"Generated #{i+1}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time")
    if i == 0 or i == 4:
        ax.set_ylabel("Amplitude")
    ax.grid(True, alpha=0.3)

# overlay plot
ax_overlay = fig.add_subplot(gs[2, 2:5])
for i in range(num_samples):
    ax_overlay.plot(generated[i, 0].cpu().numpy(), linewidth=2,
                    alpha=0.7, color=colors[i], label=f"Gen #{i+1}")
ax_overlay.plot(original[0, 0].cpu().numpy(), linewidth=3,
                color="blue", linestyle="--", alpha=0.5, label="Original")
ax_overlay.set_title("Generated Samples Overlaid", fontsize=12, fontweight="bold")
ax_overlay.set_xlabel("Time")
ax_overlay.set_ylabel("Amplitude")
ax_overlay.legend(loc="upper right", fontsize=8)
ax_overlay.grid(True, alpha=0.3)

# summary panel
ax_stats = fig.add_subplot(gs[2, 5])
ax_stats.axis("off")
stats_text = (
    "Statistics:\n\n"
    f"Training epochs: 100\n"
    f"Diffusion steps: {T}\n"
    f"Signal length: 128\n"
    f"Generated samples: {num_samples}\n\n"
    "Each generated signal\n"
    "differs because reverse\n"
    "diffusion is stochastic."
)
ax_stats.text(0.1, 0.9, stats_text, fontsize=10, verticalalignment="top",
              transform=ax_stats.transAxes, family="monospace")

plt.suptitle("1D Diffusion Model: Forward Noising and Reverse Sampling",
             fontsize=16, fontweight="bold", y=0.98)

plt.savefig("sine_diffusion_demo.png", dpi=150, bbox_inches="tight")
print("Saved visualization to 'sine_diffusion_demo.png'")
plt.show()

# Listing 6-6: A Text-to-Image Generator

In [ ]:
# --------------------------------------------------
# Step 1. Install and Import Libraries
# --------------------------------------------------
!pip install -q diffusers transformers accelerate scipy safetensors

import torch
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt

# --------------------------------------------------
# Step 2. Load the Pretrained Diffusion Pipeline
# --------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype
).to(device)

# --------------------------------------------------
# Step 3. Read a Text Prompt
# --------------------------------------------------
prompt = input("Enter description: ")

# --------------------------------------------------
# Step 4. Generate Multiple Image Variations
# --------------------------------------------------
num_images = 4
result = pipe(
    [prompt] * num_images,
    guidance_scale=7.5
)
images = result.images

# --------------------------------------------------
# Step 5. Display the Images in a Grid
# --------------------------------------------------
cols = 2
rows = (num_images + cols - 1) // cols
fig, axs = plt.subplots(rows, cols, figsize=(8, 8))

# Handle the case where matplotlib returns a 1D array
if rows == 1:
    axs = [axs]
    axs = [axs] if cols == 1 else [axs]

for i, img in enumerate(images):
    ax = axs[i // cols][i % cols] if rows > 1 else axs[0][i % cols]
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"Variation {i+1}")

# Turn off any unused axes
for j in range(i + 1, rows * cols):
    ax = axs[j // cols][j % cols] if rows > 1 else axs[0][j % cols]
    ax.axis("off")

plt.tight_layout()
plt.show()

# Listing 6-5 A Diffusion Model From Scratch on 1D Signals
This example trains a diffusion model to generate short 1D signals that look like sine waves with different amplitudes, frequencies, and phases.

In [ ]:
!pip install -q torch torchvision matplotlib

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class SineDataset(Dataset):
    def __init__(self, n=20000, length=128):
        self.n = n
        self.length = length
        self.x = torch.linspace(0, 1, steps=length)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        amp = torch.empty(1).uniform_(0.5, 1.5).item()
        freq = torch.empty(1).uniform_(1.0, 6.0).item()
        phase = torch.empty(1).uniform_(0, 2 * math.pi).item()
        y = amp * torch.sin(2 * math.pi * freq * self.x + phase)
        return y.unsqueeze(0)  # shape: [1, length]

T = 200  # diffusion steps (small for a book example)

betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)  # cumulative product

def q_sample(x0, t, noise=None):
    """
    x0: [B, 1, L]
    t:  [B] integer timesteps in [0, T-1]
    """
    if noise is None:
        noise = torch.randn_like(x0)
    a_bar = alpha_bars[t].view(-1, 1, 1)
    return torch.sqrt(a_bar) * x0 + torch.sqrt(1 - a_bar) * noise, noise

class Tiny1DUNet(nn.Module):
    def __init__(self, length=128, time_dim=64):
        super().__init__()
        self.time_emb = nn.Sequential(
            nn.Embedding(T, time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.conv1 = nn.Conv1d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv1d(64, 32, 3, padding=1)
        self.conv4 = nn.Conv1d(32, 1, 3, padding=1)

        self.fc_t = nn.Linear(time_dim, 64)  # to match conv2 channels

    def forward(self, x, t):
        # x: [B, 1, L], t: [B]
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))

        te = self.time_emb(t)              # [B, time_dim]
        te = self.fc_t(te).unsqueeze(-1)   # [B, 64, 1]
        h = h + te                         # time conditioning

        h = F.relu(self.conv3(h))
        return self.conv4(h)

dataset = SineDataset(n=20000, length=128)
loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)

model = Tiny1DUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

model.train()
for epoch in range(100):  # keep small for a demo
    total = 0.0
    for x0 in loader:
        x0 = x0.to(device)

        t = torch.randint(0, T, (x0.size(0),), device=device)
        xt, noise = q_sample(x0, t)

        pred = model(xt, t)
        loss = F.mse_loss(pred, noise)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item()

    print(f"Epoch {epoch+1}: loss={total/len(loader):.4f}")

@torch.no_grad()
def p_sample_loop(model, shape):
    model.eval()
    x = torch.randn(shape, device=device)  # start from noise

    for t in reversed(range(T)):
        tt = torch.full((shape[0],), t, device=device, dtype=torch.long)

        a = alphas[t]
        a_bar = alpha_bars[t]
        b = betas[t]

        eps = model(x, tt)

        # DDPM mean estimate
        coef1 = 1 / torch.sqrt(a)
        coef2 = b / torch.sqrt(1 - a_bar)
        mean = coef1 * (x - coef2 * eps)

        if t > 0:
            z = torch.randn_like(x)
            sigma = torch.sqrt(b)
            x = mean + sigma * z
        else:
            x = mean

    return x

samples = p_sample_loop(model, shape=(8, 1, 128)).cpu()

plt.figure(figsize=(10, 4))
for i in range(8):
    plt.plot(samples[i, 0].numpy(), alpha=0.9)
plt.title("Diffusion model samples (1D signals)")
plt.show()


In [ ]:
# Diffusion Model for 1D Signals - Reconstruction Demo
# Shows: Original → Noisy → Multiple Reconstructions

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class SineDataset(Dataset):
    def __init__(self, n=20000, length=128):
        self.n = n
        self.length = length
        self.x = torch.linspace(0, 1, steps=length)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        amp = torch.empty(1).uniform_(0.5, 1.5).item()
        freq = torch.empty(1).uniform_(1.0, 6.0).item()
        phase = torch.empty(1).uniform_(0, 2 * math.pi).item()
        y = amp * torch.sin(2 * math.pi * freq * self.x + phase)
        return y.unsqueeze(0)  # shape: [1, length]

T = 200  # diffusion steps (small for a book example)

betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)  # cumulative product

def q_sample(x0, t, noise=None):
    """
    x0: [B, 1, L]
    t:  [B] integer timesteps in [0, T-1]
    """
    if noise is None:
        noise = torch.randn_like(x0)
    a_bar = alpha_bars[t].view(-1, 1, 1)
    return torch.sqrt(a_bar) * x0 + torch.sqrt(1 - a_bar) * noise, noise

class Tiny1DUNet(nn.Module):
    def __init__(self, length=128, time_dim=64):
        super().__init__()
        self.time_emb = nn.Sequential(
            nn.Embedding(T, time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.conv1 = nn.Conv1d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv1d(64, 32, 3, padding=1)
        self.conv4 = nn.Conv1d(32, 1, 3, padding=1)

        self.fc_t = nn.Linear(time_dim, 64)  # to match conv2 channels

    def forward(self, x, t):
        # x: [B, 1, L], t: [B]
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))

        te = self.time_emb(t)              # [B, time_dim]
        te = self.fc_t(te).unsqueeze(-1)   # [B, 64, 1]
        h = h + te                         # time conditioning

        h = F.relu(self.conv3(h))
        return self.conv4(h)

dataset = SineDataset(n=20000, length=128)
loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)

model = Tiny1DUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

model.train()
for epoch in range(100):  # keep small for a demo
    total = 0.0
    for x0 in loader:
        x0 = x0.to(device)

        t = torch.randint(0, T, (x0.size(0),), device=device)
        xt, noise = q_sample(x0, t)

        pred = model(xt, t)
        loss = F.mse_loss(pred, noise)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item()

    print(f"Epoch {epoch+1}: loss={total/len(loader):.4f}")

@torch.no_grad()
def p_sample_loop(model, shape):
    model.eval()
    x = torch.randn(shape, device=device)  # start from noise

    for t in reversed(range(T)):
        tt = torch.full((shape[0],), t, device=device, dtype=torch.long)

        a = alphas[t]
        a_bar = alpha_bars[t]
        b = betas[t]

        eps = model(x, tt)

        # DDPM mean estimate
        coef1 = 1 / torch.sqrt(a)
        coef2 = b / torch.sqrt(1 - a_bar)
        mean = coef1 * (x - coef2 * eps)

        if t > 0:
            z = torch.randn_like(x)
            sigma = torch.sqrt(b)
            x = mean + sigma * z
        else:
            x = mean

    return x

# Generate samples
samples = p_sample_loop(model, shape=(5, 1, 128)).cpu()

# Get one original sample for comparison
original = dataset[0].unsqueeze(0).to(device)  # [1, 1, 128]

# Add noise to the original at a specific timestep
t_noise = torch.tensor([100], device=device)  # mid-point noise
noisy_signal, _ = q_sample(original, t_noise)
noisy_signal = noisy_signal.cpu()

# Create visualization showing: original → noisy → reconstructed samples
fig, axes = plt.subplots(7, 1, figsize=(12, 14))

# 1. Original wave
axes[0].plot(original[0, 0].cpu().numpy(), linewidth=2, color='#2E86AB')
axes[0].set_title('1. Original Wave', fontsize=12, fontweight='bold')
axes[0].set_xlim(0, 128)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-2, 2)

# 2. Noise added
axes[1].plot(noisy_signal[0, 0].numpy(), linewidth=1, color='#E63946')
axes[1].set_title('2. Noise Added (t=100)', fontsize=12, fontweight='bold')
axes[1].set_xlim(0, 128)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-2, 2)

# 3-7. Multiple reconstructed versions
for i in range(5):
    axes[i+2].plot(samples[i, 0].numpy(), linewidth=2, color='#06A77D', alpha=0.8)
    axes[i+2].set_title(f'{i+3}. Reconstructed Sample {i+1}', fontsize=12, fontweight='bold')
    axes[i+2].set_xlim(0, 128)
    axes[i+2].grid(True, alpha=0.3)
    axes[i+2].set_ylim(-2, 2)

plt.tight_layout()
plt.savefig('diffusion_reconstruction.png', dpi=150, bbox_inches='tight')
print("✅ Saved visualization to diffusion_reconstruction.png")
plt.show()

# Diffusion Model Example #2 (For Listing 6-5)

In [ ]:
"""
Image Diffusion Demo - Interactive
Opens file dialog for user to select an image and demonstrates noising/denoising process
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import os
import sys

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    from google.colab import files
except ImportError:
    IN_COLAB = False
    import tkinter as tk
    from tkinter import filedialog

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Diffusion parameters
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def load_and_preprocess_image(image_path, size=256):
    """Load image and convert to tensor"""
    try:
        img = Image.open(image_path).convert('RGB')
        img = img.resize((size, size), Image.Resampling.LANCZOS)
        img_array = np.array(img) / 255.0  # Normalize to [0, 1]
        img_tensor = torch.from_numpy(img_array).float()
        img_tensor = img_tensor.permute(2, 0, 1)  # [H, W, C] -> [C, H, W]
        img_tensor = (img_tensor - 0.5) * 2  # Normalize to [-1, 1]
        return img_tensor.unsqueeze(0).to(device)  # Add batch dimension
    except Exception as e:
        print(f"Error loading image: {e}")
        return None


def add_noise(x0, t):
    """Add noise to image at timestep t"""
    noise = torch.randn_like(x0)
    a_bar = alpha_bars[t]
    return torch.sqrt(a_bar) * x0 + torch.sqrt(1 - a_bar) * noise, noise


def tensor_to_image(tensor):
    """Convert tensor back to displayable image"""
    img = tensor.squeeze(0).cpu()
    img = (img + 1) / 2  # [-1, 1] -> [0, 1]
    img = img.permute(1, 2, 0).numpy()
    return np.clip(img, 0, 1)


class SimpleDenoiser(nn.Module):
    """Enhanced denoising network for higher resolution images"""
    def __init__(self, channels=3):
        super().__init__()

        # Time embedding
        self.time_embed = nn.Sequential(
            nn.Linear(1, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

        # Encoder with downsampling
        self.enc1 = nn.Conv2d(channels, 64, 3, padding=1)
        self.enc2 = nn.Conv2d(64, 128, 3, stride=2, padding=1)  # Downsample
        self.enc3 = nn.Conv2d(128, 256, 3, stride=2, padding=1)  # Downsample
        self.enc4 = nn.Conv2d(256, 512, 3, padding=1)

        # Time conditioning
        self.time_proj = nn.Linear(128, 512)

        # Decoder with upsampling
        self.dec1 = nn.Conv2d(512, 256, 3, padding=1)
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec2 = nn.Conv2d(256, 128, 3, padding=1)
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec3 = nn.Conv2d(128, 64, 3, padding=1)
        self.dec4 = nn.Conv2d(64, channels, 3, padding=1)

    def forward(self, x, t):
        # Encode with downsampling
        h1 = F.relu(self.enc1(x))
        h2 = F.relu(self.enc2(h1))
        h3 = F.relu(self.enc3(h2))
        h4 = F.relu(self.enc4(h3))

        # Time conditioning
        t_emb = self.time_embed(t.float().unsqueeze(-1) / T)
        t_emb = self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h4 = h4 + t_emb

        # Decode with upsampling
        h = F.relu(self.dec1(h4))
        h = self.up1(h)
        h = F.relu(self.dec2(h))
        h = self.up2(h)
        h = F.relu(self.dec3(h))
        return self.dec4(h)


def simple_denoise(noisy_img, model, steps=50):
    """DDPM denoising algorithm"""
    model.eval()
    x = noisy_img.clone()

    with torch.no_grad():
        step_size = T // steps
        timesteps = list(range(T - 1, -1, -step_size))

        for i, t_val in enumerate(timesteps):
            t = torch.tensor([t_val], device=device)

            # Predict noise
            pred_noise = model(x, t)

            # Get alpha values
            alpha_t = alphas[t_val]
            alpha_bar_t = alpha_bars[t_val]

            # Calculate coefficients
            if t_val > 0:
                alpha_bar_prev = alpha_bars[t_val - step_size] if t_val >= step_size else alpha_bars[0]
            else:
                alpha_bar_prev = torch.tensor(1.0, device=device)

            # Predict x0 from xt and predicted noise
            pred_x0 = (x - torch.sqrt(1 - alpha_bar_t) * pred_noise) / torch.sqrt(alpha_bar_t)
            pred_x0 = torch.clamp(pred_x0, -1, 1)

            if t_val > 0:
                # Calculate direction pointing to xt
                dir_xt = torch.sqrt(1 - alpha_bar_prev) * pred_noise

                # Calculate xt-1
                x = torch.sqrt(alpha_bar_prev) * pred_x0 + dir_xt

                # Add noise for stochasticity (except last step)
                if i < len(timesteps) - 1:
                    noise = torch.randn_like(x) * 0.1
                    x = x + noise
            else:
                x = pred_x0

            # Clamp to valid range
            x = torch.clamp(x, -1, 1)

    return x


def train_model(model, original_img, epochs=3000, batch_size=16):
    """Train the model on the input image to improve reconstruction quality"""
    print("\n🔧 Training model on your image...")
    print(f"   Epochs: {epochs}, Batch size: {batch_size}")

    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        total_loss = 0

        for _ in range(batch_size):
            # Random timestep
            t = torch.randint(0, T, (1,), device=device)

            # Add noise
            noisy_img, noise = add_noise(original_img, t)

            # Predict noise
            pred_noise = model(noisy_img, t)

            # Loss
            loss = F.mse_loss(pred_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()

        if (epoch + 1) % 50 == 0:
            avg_loss = total_loss / batch_size
            lr = optimizer.param_groups[0]['lr']
            print(f"   Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}, LR: {lr:.6f}")

    print("✅ Training complete!")
    model.eval()
    return model


def visualize_process(original_img, save_path='image_diffusion_demo.png'):
    """Visualize the noising and denoising process with multiple reconstructions"""

    # Create noisy versions at different timesteps
    timesteps = [0, 200, 400, 600, 800, 999]
    noisy_images = []

    for t in timesteps:
        t_tensor = torch.tensor([t], device=device)
        noisy, _ = add_noise(original_img, t_tensor)
        noisy_images.append(noisy)

    # Initialize and train the model
    model = SimpleDenoiser().to(device)
    model = train_model(model, original_img, epochs=100, batch_size=8)

    # Create multiple different reconstructions from the same noisy image
    heavily_noised = noisy_images[-2]  # t=800
    reconstruction1 = simple_denoise(heavily_noised.clone(), model, steps=20)
    reconstruction2 = simple_denoise(heavily_noised.clone(), model, steps=20)
    reconstruction3 = simple_denoise(heavily_noised.clone(), model, steps=20)

    # Create visualization with 4 rows
    fig, axes = plt.subplots(4, 3, figsize=(12, 16))

    # Row 1: Original and progressive noising
    axes[0, 0].imshow(tensor_to_image(original_img))
    axes[0, 0].set_title('Original Image', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(tensor_to_image(noisy_images[2]))  # t=400
    axes[0, 1].set_title('Light Noise (t=400)', fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(tensor_to_image(noisy_images[4]))  # t=800
    axes[0, 2].set_title('Heavy Noise (t=800)', fontsize=12, fontweight='bold')
    axes[0, 2].axis('off')

    # Row 2: More noising stages
    axes[1, 0].imshow(tensor_to_image(noisy_images[1]))  # t=200
    axes[1, 0].set_title('Very Light Noise (t=200)', fontsize=12, fontweight='bold')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(tensor_to_image(noisy_images[3]))  # t=600
    axes[1, 1].set_title('Medium Noise (t=600)', fontsize=12, fontweight='bold')
    axes[1, 1].axis('off')

    axes[1, 2].imshow(tensor_to_image(noisy_images[5]))  # t=999
    axes[1, 2].set_title('Pure Noise (t=999)', fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')

    # Row 3: Starting point for reconstructions
    axes[2, 0].imshow(tensor_to_image(heavily_noised))
    axes[2, 0].set_title('Noisy Input (t=800)', fontsize=12, fontweight='bold')
    axes[2, 0].axis('off')

    axes[2, 1].imshow(tensor_to_image(original_img))
    axes[2, 1].set_title('Target (Original)', fontsize=12, fontweight='bold')
    axes[2, 1].axis('off')

    axes[2, 2].text(0.5, 0.5, 'Different\nReconstructions\nfrom Same\nNoisy Input →',
                    ha='center', va='center', fontsize=14, fontweight='bold',
                    transform=axes[2, 2].transAxes)
    axes[2, 2].axis('off')

    # Row 4: Multiple different reconstructions
    axes[3, 0].imshow(tensor_to_image(reconstruction1))
    axes[3, 0].set_title('Reconstruction #1\n(Trained Model)', fontsize=12, fontweight='bold')
    axes[3, 0].axis('off')

    axes[3, 1].imshow(tensor_to_image(reconstruction2))
    axes[3, 1].set_title('Reconstruction #2\n(Trained Model)', fontsize=12, fontweight='bold')
    axes[3, 1].axis('off')

    axes[3, 2].imshow(tensor_to_image(reconstruction3))
    axes[3, 2].set_title('Reconstruction #3\n(Trained Model)', fontsize=12, fontweight='bold')
    axes[3, 2].axis('off')

    plt.suptitle('Image Diffusion: Noising and Multiple Reconstructions',
                 fontsize=16, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"\n✅ Saved visualization to {save_path}")
    plt.show()


def select_image_file():
    """Open file dialog to select an image (Colab-compatible)"""
    if IN_COLAB:
        # Google Colab: use file upload widget
        print("\n📁 Please upload an image file...")
        print("   Click 'Choose Files' button to upload")
        print("   Supported formats: JPG, JPEG, PNG, BMP, GIF, TIFF, WebP, HEIC")
        print("   Or skip upload to use a sample pattern")

        try:
            uploaded = files.upload()

            if uploaded:
                # Get the first uploaded file
                filename = list(uploaded.keys())[0]
                print(f"✅ Uploaded: {filename}")
                return filename
            else:
                print("⚠️  No file uploaded")
                return ""
        except Exception as e:
            print(f"⚠️  Upload cancelled or failed: {e}")
            return ""
    else:
        # Local environment: use tkinter file dialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)

        print("\n📁 Opening file dialog...")
        print("   Please select an image file (JPG, PNG, etc.)")
        print("   Or click 'Cancel' to use a sample pattern")

        # Open file dialog
        file_path = filedialog.askopenfilename(
            title="Select an Image",
            filetypes=[
                ("All Image files", "*.jpg *.jpeg *.png *.bmp *.gif *.tiff *.tif *.webp *.heic *.heif"),
                ("JPEG/JPG files", "*.jpg *.jpeg *.jpe *.jfif"),
                ("PNG files", "*.png"),
                ("BMP files", "*.bmp *.dib"),
                ("GIF files", "*.gif"),
                ("TIFF files", "*.tiff *.tif"),
                ("WebP files", "*.webp"),
                ("HEIC/HEIF files", "*.heic *.heif"),
                ("All files", "*.*")
            ]
        )

        root.destroy()
        return file_path


def main():
    print("🎨 Image Diffusion Demo - Interactive")
    print("=" * 60)
    print("This demo shows how diffusion models add and remove noise")
    print("=" * 60)

    # Open file dialog to select image
    image_path = select_image_file()

    if not image_path or image_path == "":
        # Create a simple pattern if no image selected (user clicked Cancel)
        print("\n🎨 No image selected. Creating sample pattern...")
        size = 256
        x = torch.linspace(-1, 1, size)
        y = torch.linspace(-1, 1, size)
        xx, yy = torch.meshgrid(x, y, indexing='ij')

        # Create a colorful pattern
        r = torch.sin(3 * xx) * torch.cos(3 * yy)
        g = torch.cos(2 * xx) * torch.sin(2 * yy)
        b = torch.sin(xx + yy) * torch.cos(xx - yy)

        pattern = torch.stack([r, g, b], dim=0)
        pattern = (pattern + 1) / 2  # [0, 1]
        pattern = (pattern - 0.5) * 2  # [-1, 1]
        original_img = pattern.unsqueeze(0).to(device)
        print("✅ Sample pattern created")
    else:
        # Load user's image
        print(f"\n📥 Loading image from: {image_path}")
        original_img = load_and_preprocess_image(image_path)

        if original_img is None:
            print("❌ Failed to load image. Using sample pattern instead.")
            # Fallback to pattern
            size = 256
            x = torch.linspace(-1, 1, size)
            y = torch.linspace(-1, 1, size)
            xx, yy = torch.meshgrid(x, y, indexing='ij')
            r = torch.sin(3 * xx) * torch.cos(3 * yy)
            g = torch.cos(2 * xx) * torch.sin(2 * yy)
            b = torch.sin(xx + yy) * torch.cos(xx - yy)
            pattern = torch.stack([r, g, b], dim=0)
            pattern = (pattern + 1) / 2
            pattern = (pattern - 0.5) * 2
            original_img = pattern.unsqueeze(0).to(device)
        else:
            print("✅ Image loaded successfully")

    # Show image info
    print(f"\n📊 Image shape: {original_img.shape}")
    print(f"📱 Device: {device}")

    # Create visualization
    print("\n🎨 Creating diffusion visualization...")
    print("   This will show:")
    print("   - Original image")
    print("   - Progressive noise addition (6 stages)")
    print("   - Multiple different reconstructions from the same noisy input")
    print("   - Demonstrates stochastic nature of diffusion models")

    visualize_process(original_img)

    print("\n" + "=" * 60)
    print("✨ Demo complete!")
    print("📁 Check 'image_diffusion_demo.png' for results")
    print("=" * 60)
    print("\n💡 The model was trained on your image for better reconstructions.")
    print("   Each reconstruction is different due to the stochastic sampling process,")
    print("   but all should closely resemble the original image!")


if __name__ == '__main__':
    main()

# Made with Bob
